In [1]:
import os
import cv2
import torch
import ollama
from groundingdino.util.inference import load_model, load_image, predict

In [3]:
# 1. Configuration
DATASETS_ROOT = "./Datasets"
CLASS_MAP = {"Helmet": 0, "Pagdi": 1, "No-Helmet": 2}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# Load DINO Model
dino_model = load_model("./VLM-Pipeline/GroundingDINO_SwinT_OGC.py", "./VLM-Pipeline/groundingdino_swint_ogc.pth")

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["BUILD_WITH_CUDA"] = "0"
device = torch.device("cpu")
dino_model = dino_model.to(device)

DINO_PROMPT = "motorcycle rider head"

final text_encoder_type: bert-base-uncased


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [33]:
for img_name in os.listdir(IMAGE_DIR):
    img_path = os.path.join(IMAGE_DIR, img_name)
    image_source, image = load_image(img_path)
    
    # 1. Zero-Shot Bounding Boxes
    boxes, logits, phrases = predict(
        model=dino_model,
        image=image,
        caption=DINO_PROMPT,
        box_threshold=0.3,
        text_threshold=0.25,
        device=device
    )
    
    txt_filename = os.path.splitext(img_name)[0] + ".txt"
    txt_filepath = os.path.join(LABEL_DIR, txt_filename)
    
    with open(txt_filepath, 'w') as f:
        h, w, _ = image_source.shape
        
        for box in boxes:
            cx, cy, bw, bh = box.numpy()
            
            # Crop for VLM using the original, full head box
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            
            crop = image_source[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
            crop_path = "temp_crop.jpg"
            cv2.imwrite(crop_path, crop)
            
            # 2. Strict VLM Classification (Temperature = 0)
            vlm_prompt = "Classify the headgear in this image into one of three strict categories: 'Helmet', 'Pagdi', or 'No-Helmet'. Reply with only the exact category name. Do not include any other words."
            
            response = ollama.generate(
                model='llava',
                prompt=vlm_prompt,
                images=[crop_path],
                options={'temperature': 0.0} # THIS KILLS HALLUCINATIONS
            )
            
            pred_class = response['response'].strip().replace("'", "").replace('"', '')
            
            # Parse the output safely
            if "Helmet" in pred_class and "No" not in pred_class:
                final_class = "Helmet"
            elif "Pagdi" in pred_class or "Turban" in pred_class:
                final_class = "Pagdi"
            else:
                final_class = "No-Helmet"
                
            class_id = CLASS_MAP[final_class]
            
            # 3. The Geometric Heuristic (Fixing the YOLO Box)
            final_cx, final_cy, final_bw, final_bh = cx, cy, bw, bh
            
            # If it is NOT a helmet, chop off the bottom part of the box
            if class_id in [1, 2]: 
                # Keep only the top 60% of the original height
                new_bh = bh * 0.60 
                
                # Shift the Y-center up by half of the removed height
                removed_height = bh * 0.40
                new_cy = cy - (removed_height / 2)
                
                final_bh = new_bh
                final_cy = new_cy
            
            # 4. Write perfectly cropped YOLO Format
            f.write(f"{class_id} {final_cx:.6f} {final_cy:.6f} {final_bw:.6f} {final_bh:.6f}\n")

print("Refined Auto-annotation complete.")

FileNotFoundError: [Errno 2] No such file or directory: './Datasets/gender.multiclass-hijab-200/data'

In [4]:
for dataset_name in os.listdir(DATASETS_ROOT):
    dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
    
    # Skip if it's not a directory (e.g., a stray file in the datasets root)
    if not os.path.isdir(dataset_path):
        continue
        
    img_dir = os.path.join(dataset_path, "data")
    lbl_dir = os.path.join(dataset_path, "labels")
    
    # Ensure the directories actually exist to avoid crashes
    if not os.path.exists(img_dir):
        print(f"Skipping {dataset_name}: No 'data' folder found.")
        continue
        
    # Auto-create the labels folder if you forgot to make it
    os.makedirs(lbl_dir, exist_ok=True)
    print(f"\n--- Processing Dataset: {dataset_name} ---")

    # 3. Iterate through images
    for file_name in os.listdir(img_dir):
        # ERROR PREVENTION: Check file extension before doing anything
        ext = os.path.splitext(file_name)[1].lower()
        if ext not in VALID_EXTENSIONS:
            print(f"  -> Skipping non-image file: {file_name}")
            continue

        img_path = os.path.join(img_dir, file_name)
        
        try:
            image_source, image = load_image(img_path)
            
            # Zero-Shot Bounding Boxes
            boxes, logits, phrases = predict(
                model=dino_model,
                image=image,
                caption=DINO_PROMPT,
                box_threshold=0.3,
                text_threshold=0.25,
                device=device
            )
            
            txt_filename = os.path.splitext(file_name)[0] + ".txt"
            txt_filepath = os.path.join(lbl_dir, txt_filename)
            
            with open(txt_filepath, 'w') as f:
                h, w, _ = image_source.shape
                
                for box in boxes:
                    cx, cy, bw, bh = box.numpy()
                    
                    # Crop for VLM using the original full head box
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    
                    # Failsafe for boxes that clip outside the image boundary
                    crop = image_source[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
                    
                    # Skip if crop is completely empty/invalid
                    if crop.size == 0:
                        continue
                        
                    crop_path = "temp_crop.jpg"
                    cv2.imwrite(crop_path, crop)
                    
                    # Strict VLM Classification (Temperature = 0)
                    vlm_prompt = "Classify the headgear in this image into one of three strict categories: 'Helmet', 'Pagdi', or 'No-Helmet'. Reply with only the exact category name. Do not include any other words."
                    
                    response = ollama.generate(
                        model='llava',
                        prompt=vlm_prompt,
                        images=[crop_path],
                        options={'temperature': 0.0}
                    )
                    
                    pred_class = response['response'].strip().replace("'", "").replace('"', '')
                    
                    # Parse the output safely
                    if "Helmet" in pred_class and "No" not in pred_class:
                        final_class = "Helmet"
                    elif "Pagdi" in pred_class or "Turban" in pred_class:
                        final_class = "Pagdi"
                    else:
                        final_class = "No-Helmet"
                        
                    class_id = CLASS_MAP[final_class]
                    
                    # The Geometric Heuristic
                    final_cx, final_cy, final_bw, final_bh = cx, cy, bw, bh
                    
                    if class_id in [1, 2]: 
                        new_bh = bh * 0.60 
                        removed_height = bh * 0.40
                        new_cy = cy - (removed_height / 2)
                        
                        final_bh = new_bh
                        final_cy = new_cy
                    
                    # Write YOLO Format
                    f.write(f"{class_id} {final_cx:.6f} {final_cy:.6f} {final_bw:.6f} {final_bh:.6f}\n")
                    
            print(f"Processed: {file_name}")

        except Exception as e:
            print(f"  -> ERROR processing {file_name}: {e}")

print("\nAll datasets processed successfully.")


--- Processing Dataset: Random-images-no-demographic-800 ---


KeyboardInterrupt: 